In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install -U "bitsandbytes>=0.46.1" accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 63.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.16.1
    Uninstalling transformers-5.16.1:
      Successfully uninstalled transformers-5.16.1
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.14.0
    Uninstalling accelerate-1.14.0:
      Successfully uninstalled accelerate-1.14.0


In [ ]:
import torch
from transformers import BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration, AutoProcessor
from PIL import Image

model_name = "Qwen/Qwen2.5-VL-7B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_name,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float16
)

processor = AutoProcessor.from_pretrained(model_name)

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/729 [00:00<?, ?it/s]

...

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [3]:
import os

base_dir = "/content/drive/MyDrive/Colab Notebooks/RegDia_pages"

issue_folders = sorted(
    [
        folder
        for folder in os.listdir(base_dir)
        if os.path.isdir(
            os.path.join(base_dir, folder)
        )
    ]
)

print(f"Found {len(issue_folders)} issue folders.")

for folder in issue_folders:
    print(folder)

Found 0 issue folders.


In [ ]:
def resize_to_width(image, width=1024):

    original_width, original_height = image.size

    scale = width / original_width

    new_height = int(original_height * scale)

    return image.resize(
        (width, new_height),
        Image.Resampling.LANCZOS
    )

In [ ]:
import cv2
import numpy as np


def detect_split_positions(
    image,
    target_ratios=(1/3, 2/3),
    tolerance=0.25,
    min_tile_ratio=0.20
):


    # Convert image to grayscale

    gray = cv2.cvtColor(
        np.array(image),
        cv2.COLOR_RGB2GRAY
    )

    # Binarize

    _, binary = cv2.threshold(
        gray,
        0,
        255,
        cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
    )


    # Horizontal projection

    horizontal_projection = np.sum(
        binary > 0,
        axis=1
    )

    # Detect rows containing text

    threshold = 0.01 * binary.shape[1]

    text_rows = (
        horizontal_projection > threshold
    )

    # Find continuous text regions

    regions = []

    in_region = False
    start = None

    for y, has_text in enumerate(text_rows):

        if has_text and not in_region:

            start = y
            in_region = True

        elif not has_text and in_region:

            end = y - 1

            regions.append(
                (start, end)
            )

            in_region = False

    if in_region:

        regions.append(
            (start, len(text_rows) - 1)
        )

    # Calculate gaps between text lines

    gaps = []

    for i in range(len(regions) - 1):

        current_end = regions[i][1]
        next_start = regions[i + 1][0]

        gap = next_start - current_end

        center = (
            current_end +
            next_start
        ) / 2

        gaps.append({
            "after_line": i,
            "y_start": current_end,
            "y_end": next_start,
            "size": gap,
            "center": center
        })

    # Find candidate gaps for each target

    height = image.height

    selected = []

    for target_ratio in target_ratios:

        target_y = target_ratio * height

        candidates = []

        for gap in gaps:

            center = gap["center"]

            ratio = center / height

            distance = abs(
                ratio - target_ratio
            )

            # Candidate must be within tolerance
            if distance <= tolerance:

                candidates.append(
                    (gap, distance)
                )


        # Rank candidates
        #
        # We want:
        #   - close to target
        #   - large whitespace gap
        #
        # Distance is the primary criterion.
        # Gap size breaks ties.

        if candidates:

            candidates.sort(
                key=lambda x: (
                    x[1],
                    -x[0]["size"]
                )
            )

            best_gap = candidates[0][0]

            selected.append(
                best_gap
            )

        else:

            # No suitable gap found
            #
            # Temporarily use the target position as fallback.

            fallback_y = int(
                target_y
            )

            selected.append({
                "after_line": None,
                "y_start": fallback_y,
                "y_end": fallback_y,
                "size": 0,
                "center": fallback_y,
                "fallback": True
            })

    # Extract split positions

    split_positions = [
        int(gap["center"])
        for gap in selected
    ]

    split_positions.sort()

    # Make sure splits are different

    min_tile_size = int(
        height * min_tile_ratio
    )

    first = split_positions[0]
    second = split_positions[1]

    # If the two splits are identical or too close,
    # force them towards 1/3 and 2/3.

    if (
        second - first
        < min_tile_size
    ):

        print(
            "⚠ Splits too close together."
            " Applying fallback."
        )

        first = int(
            height / 3
        )

        second = int(
            2 * height / 3
        )

    # Make sure first split isn't too close to bottom/top

    if first < min_tile_size:

        print(
            "⚠ First split too high."
            " Using 1/3 fallback."
        )

        first = int(
            height / 3
        )

    if (
        height - second
        < min_tile_size
    ):

        print(
            "⚠ Second split too low."
            " Using 2/3 fallback."
        )

        second = int(
            2 * height / 3
        )

    # Final validation

    if not (
        0 < first < second < height
    ):

        print(
            "⚠ Invalid split configuration."
            " Using 1/3 and 2/3 fallback."
        )

        first = int(
            height / 3
        )

        second = int(
            2 * height / 3
        )

    return [
        first,
        second
    ]

In [ ]:
def create_tiles(image, split_positions, tile_width=1024):

    width, height = image.size

    boundaries = (
        [0]
        + sorted(split_positions)
        + [height]
    )

    tiles = []

    for i in range(len(boundaries) - 1):

        y1 = boundaries[i]
        y2 = boundaries[i + 1]

        # Crop original image
        tile = image.crop(
            (0, y1, width, y2)
        )

        # Resize tile
        tile = resize_to_width(
            tile,
            width=tile_width
        )

        tiles.append(tile)

    return tiles

In [ ]:
import os
import re
import torch
from PIL import Image


def get_canvas_number(filename):
    """
    Extract canvas number from filenames such as:

        bsb11130457_00195.jpg

    Returns:
        195
    """

    match = re.search(
        r'_(\d+)\.jpg$',
        filename,
        re.IGNORECASE
    )

    if not match:
        raise ValueError(
            f"Could not extract canvas number from: {filename}"
        )

    return int(match.group(1))


def get_issue_metadata(issue_folder):
    """
    Extract issue number and date from folder names such as:

        Num_1_02_01_1770
        Num_52_24_12_1770

    Returns:
        issue_number, issue_date
    """

    match = re.match(
        r'Num_(\d+)_(\d{2})_(\d{2})_(\d{4})$',
        issue_folder
    )

    if not match:
        raise ValueError(
            f"Could not extract issue metadata from: "
            f"{issue_folder}"
        )

    issue_number = int(match.group(1))
    day = match.group(2)
    month = match.group(3)
    year = match.group(4)

    issue_date = f"{year}-{month}-{day}"

    return issue_number, issue_date

In [ ]:
def transcribe_image(
    image,
    canvas_number,
    model,
    processor,
    max_new_tokens=800
):

    prompt = """
You are an OCR system for historical German documents (18th–19th century Fraktur print).

You are transcribing a historical printed German diary page.

Output raw transcription only.
DO NOT REPEAT THE PROMPT OR RETURN ANYTHING OTHER THAN THE TRANSCRIPTION.

Your task is to transcribe the entire page exactly as it appears.

Make sure to follow these rules:
- Do NOT summarize
- Do NOT interpret meaning
- Do NOT correct spelling
- Preserve line breaks as in the image
- Preserve original spelling
- If a word is unclear, mark it as [unclear]

IMPORTANT:
- The page contains multiple separate entries.
- Transcribe strictly line-by-line.
- Preserve section structure.
- Do NOT merge or reorganize text.
- Do NOT correct spelling.

OUTPUT:
Return ONLY the raw transcription text.
DO NOT REPEAT THE PROMPT OR RETURN ANYTHING OTHER THAN THE TRANSCRIPTION.
"""

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt",
        padding=True
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            use_cache=True
        )

    # Remove prompt from generated sequence
    input_length = inputs["input_ids"].shape[1]

    generated_ids = output_ids[:, input_length:]

    output = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    # Free GPU memory
    del inputs
    del output_ids
    del generated_ids

    torch.cuda.empty_cache()

    return output.strip()

In [ ]:
def process_page(
    image_path,
    model,
    processor,
    tile_width=1024
):

    filename = os.path.basename(image_path)

    # Extract canvas number from filename
    canvas_number = get_canvas_number(filename)

    image = Image.open(
        image_path
    ).convert("RGB")

    # Detect natural split positions
    split_positions = detect_split_positions(
        image
    )

    # Create tiles
    tiles = create_tiles(
        image,
        split_positions,
        tile_width=tile_width
    )

    # Transcribe each tile
    transcriptions = []

    for tile in tiles:

        transcription = transcribe_image(
            tile,
            canvas_number,
            model,
            processor
        )

        transcriptions.append(
            transcription
        )

    # Combine tile transcriptions
    combined_transcription = "\n\n".join(
        transcriptions
    )

    return (
        canvas_number,
        combined_transcription,
        split_positions
    )

In [ ]:
import os

# Directories

base_dir = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "RegDia_pages"
)

output_base = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "RegDia_transcriptions"
)

os.makedirs(
    output_base,
    exist_ok=True
)

# Find all issue folders

issue_folders = sorted(
    [
        folder
        for folder in os.listdir(base_dir)
        if os.path.isdir(
            os.path.join(base_dir, folder)
        )
    ]
)

print(
    f"Found {len(issue_folders)} issue folders."
)


# Process every issue

for issue_folder in issue_folders:

    # Extract issue number and date

    try:

        issue_number, issue_date = (
            get_issue_metadata(issue_folder)
        )

    except ValueError as e:

        print(f"\n⚠ {e}")
        print("Skipping folder.")
        continue


    issue_path = os.path.join(
        base_dir,
        issue_folder
    )

    # Create corresponding output folder

    issue_output_dir = os.path.join(
        output_base,
        issue_folder
    )

    os.makedirs(
        issue_output_dir,
        exist_ok=True
    )

    # Find all JPG pages

    image_files = sorted(
        [
            file
            for file in os.listdir(issue_path)
            if file.lower().endswith(".jpg")
        ]
    )


    print(
        f"\n{'=' * 70}"
    )

    print(
        f"Issue {issue_number} | "
        f"Date: {issue_date}"
    )

    print(
        f"Folder: {issue_folder}"
    )

    print(
        f"Pages: {len(image_files)}"
    )

    print(
        f"{'=' * 70}"
    )

    # Process every page

    for filename in image_files:

        image_path = os.path.join(
            issue_path,
            filename
        )

        # Same filename, but .txt instead of .jpg
        output_filename = os.path.splitext(
            filename
        )[0] + ".txt"

        output_path = os.path.join(
            issue_output_dir,
            output_filename
        )


        # Skip already processed pages

        if os.path.exists(output_path):

            print(
                f"↪ Skipping {filename} "
                f"(already exists)"
            )

            continue


        # Process page

        try:

            (
                canvas_number,
                transcription,
                split_positions
            ) = process_page(
                image_path,
                model,
                processor,
                tile_width=1024
            )


            # Create metadata header

            metadata = (
                f"Issue number: {issue_number}\n"
                f"Issue date: {issue_date}\n"
                f"Canvas: {canvas_number}\n"
                f"\n"
                f"--- TRANSCRIPTION ---\n\n"
            )

            # Save transcription

            with open(
                output_path,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(
                    metadata +
                    transcription
                )


            print(
                f"✓ Finished {filename} "
                f"(Canvas {canvas_number})"
            )


        except Exception as e:

            print(
                f"\n✗ ERROR processing "
                f"{filename}:"
            )

            print(e)

Found 52 issue folders.

Issue 10 | Date: 1770-03-06
Folder: Num_10_06_03_1770
Pages: 8
↪ Skipping bsb11130457_00079.jpg (already exists)
↪ Skipping bsb11130457_00080.jpg (already exists)
↪ Skipping bsb11130457_00081.jpg (already exists)
↪ Skipping bsb11130457_00082.jpg (already exists)
↪ Skipping bsb11130457_00083.jpg (already exists)
↪ Skipping bsb11130457_00084.jpg (already exists)
↪ Skipping bsb11130457_00085.jpg (already exists)
↪ Skipping bsb11130457_00086.jpg (already exists)

Issue 11 | Date: 1770-03-13
Folder: Num_11_13_03_1770
Pages: 8
↪ Skipping bsb11130457_00087.jpg (already exists)
↪ Skipping bsb11130457_00088.jpg (already exists)
↪ Skipping bsb11130457_00089.jpg (already exists)
↪ Skipping bsb11130457_00090.jpg (already exists)
↪ Skipping bsb11130457_00091.jpg (already exists)
↪ Skipping bsb11130457_00092.jpg (already exists)
↪ Skipping bsb11130457_00093.jpg (already exists)
↪ Skipping bsb11130457_00094.jpg (already exists)

Issue 12 | Date: 1770-03-20
Folder: Num_12_20_0

/usr/local/lib/python3.13/dist-packages/bitsandbytes/backends/cuda/ops.py:957: UserWarning: inner dimension (3420) is not aligned for fast kernel with blocksize=64, falling back to slower implementation.
  warn(


✓ Finished bsb11130457_00077.jpg (Canvas 77)
✓ Finished bsb11130457_00078.jpg (Canvas 78)


In [ ]:
import os
import re
import json
import gc
import torch

from json_repair import repair_json

# PATHS

transcription_base = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "RegDia_transcriptions"
)

entity_base = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "RegDia_entities"
)

# 8 pages at once requires too much gpu
CHUNK_SIZE = 4

# HELPER FUNCTIONS

def get_issue_number(folder_name):

    match = re.match(
        r"Num_(\d+)_",
        folder_name
    )

    if not match:
        raise ValueError(
            f"Could not extract issue number from: "
            f"{folder_name}"
        )

    return int(match.group(1))


def get_canvas_number(filename):

    match = re.search(
        r"_(\d+)\.txt$",
        filename
    )

    if not match:
        raise ValueError(
            f"Could not extract canvas number from: "
            f"{filename}"
        )

    return int(match.group(1))


def get_transcription_metadata(text):

    issue_number_match = re.search(
        r"Issue number:\s*(\d+)",
        text
    )

    issue_date_match = re.search(
        r"Issue date:\s*(\d{4}-\d{2}-\d{2})",
        text
    )

    canvas_match = re.search(
        r"Canvas:\s*(\d+)",
        text
    )

    issue_number = (
        int(issue_number_match.group(1))
        if issue_number_match
        else None
    )

    issue_date = (
        issue_date_match.group(1)
        if issue_date_match
        else None
    )

    canvas = (
        int(canvas_match.group(1))
        if canvas_match
        else None
    )

    return (
        issue_number,
        issue_date,
        canvas
    )


def load_transcription(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return f.read()


# PROMPT

EXTRACTION_PROMPT = r"""
```text
You are extracting structured information from a historical
German newspaper/diary transcription from the Regensburger
Diarium.

The input below is a TRANSCRIPTION of one or more historical
pages.

Your task is to extract ALL relevant information from the
ENTIRE transcription and return it as JSON.

IMPORTANT:
You must process the complete transcription from beginning to
end.

Do NOT focus only on one section.

If information from evangelisch, catholisch, or
arrivals_departures is present in the transcription, include it
in the corresponding JSON section.

If a section is not present in the transcription, return an
empty list for that section.

============================================================
VERY IMPORTANT: EXAMPLE VALUES
============================================================

The JSON structure shown below is ONLY an example of the
required structure.

The example values are NOT data from the transcription.

NEVER copy example names, dates, occupations, parishes,
travellers, places, gates, transports, lodgings, or any other
example values into the output.

Only use information that actually occurs in the transcription.

============================================================
EVANGELISCH / CATHOLISCH
============================================================

The transcription may contain sections for:

- evangelisch
- catholisch

Each section can contain one or more parishes.

Each parish has its own name.

For every parish, extract:

- baptised
- buried
- deaths

For every event extract:

DATE:
- verbatim: preserve the date as written
- normalised: normalize only when it can be determined reliably
- otherwise use null

PERSONS:
- role
- given_name
- family_name
- occupation_verbatim
- occupation_normalised

If an event includes multiple different persons, they must remain separate.
All three event types can have multiple persons, like a child and their father.
If no other persons are mentioned you DO NOT need to create two persons, in this case just create one subject.

For example:

- subject = child
- father = father of the child

Do NOT mix information between these persons.
Check the entire baptims entry, the subject (child) usually has some fields missing like occupation. If only one occupation is given then it belongs to the father not the child.

Burials and Deaths sometimes only have ONE person, the subject.
Only create a second person, for example "father", if a father is mentioned, otherwise only create one person, the "subject", if there is only one person mentioned for an entry.

IMPORTANT: DO NOT DUPLICATE ANYTHING, in most cases a person is only mentioned once. Either they were baptised, buried or died, and only in ONE parish.
DO NOT USE A PERSON TWICE unless the transcription actually mentions them twice.

Only split given name and family name when reasonably clear.

If the name cannot be reliably split, use the available part
and null for the other part.

OCCUPATIONS:
- occupation_verbatim must preserve the historical wording
- occupation_normalised should normally be null

AGE:
- verbatim = original historical age expression
- age_days = age converted to days only when it can be
  determined reliably
- otherwise age_days = null

============================================================
ARRIVALS AND DEPARTURES
============================================================

Extract traveller information from arrivals and departures.

For every actual traveller record, extract:

- date
- gate
- direction
- person
- time
- transport
- origin
- lodging
- s_value

The person contains:

- given_name
- family_name
- occupation_verbatim
- occupation_normalised

Keep categories separate.

For example:

- "Vormittags" belongs in "time"
- "Nachmittags" belongs in "time"
- "Postwagen" belongs in "transport"
- "Kutsche" belongs in "transport"
- "Passau" belongs in "origin"
- "Schiffmeister" belongs in "occupation_verbatim"

Do not mix names, occupations, origins, times, transports, or
lodgings.
Make sure to seperate each information into it's corresponding field in the JSON
- "Juriste von Passau" means "Juriste" belongs in "occupation_verbatim and "Passau" belongs in "origin"
- "von X" or "aus X" is usually not part of a persons name, and instead means that they came from ("von") X where "X" would be the origin

Origins usually are german towns, villages, etc. but they may also be states or countries
- "Handelsmann aus Holland" means "Handelsmann" belongs in "occupation_verbatim" and "Holland" belongs in "origin"

Do not create an empty traveller record containing only a date.

============================================================
LEGITIMATIONES AND AVERTISSEMENTS
============================================================

The transcription may contain:

- Legitimationes
- AVERTISSEMENTS

Only extract information from these sections if it clearly fits
into the existing JSON structure.

For example, an additional traveller can be added to
arrivals_departures.

Otherwise ignore the information.

============================================================
GENERAL RULES
============================================================

- Process the ENTIRE transcription.
- Extract ALL relevant information.
- Do not return only one section if other sections are present.
- Return ONLY valid JSON.
- Do not return markdown.
- Do not return explanations.
- Do not add text before or after the JSON.
- Do not invent information.
- If information is uncertain, use null.
- Preserve historical wording where requested.
- Do not mix information belonging to different persons.
- Do not copy information from this example.
- The transcription is the ONLY source of extracted data.

============================================================
REQUIRED JSON STRUCTURE
============================================================

Return exactly these three top-level extraction sections:

{
    "evangelisch": {
        "parishes": []
    },

    "catholisch": {
        "parishes": []
    },

    "arrivals_departures": []
}

The arrays must contain the extracted information when it is
present in the transcription.

If no information is present for a section, leave its array
empty.

For example, if the transcription contains evangelical parish
information but no catholic information, the result should have
the form:

{
    "evangelisch": {
        "parishes": [
            {
                "name": "actual parish name from transcription",
                "baptised": [],
                "buried": [],
                "deaths": []
            }
        ]
    },

    "catholisch": {
        "parishes": []
    },

    "arrivals_departures": []
}

The name and all other values in this example are illustrative.
They must NOT be copied unless they actually occur in the
transcription.

For an actual parish, use this structure:

{
    "name": null,

    "baptised": [
        {
            "date": {
                "verbatim": null,
                "normalised": null
            },

            "persons": [
                {
                    "role": "subject",
                    "given_name": null,
                    "family_name": null,
                    "occupation_verbatim": null,
                    "occupation_normalised": null
                },
                {
                    "role": "father",
                    "given_name": null,
                    "family_name": null,
                    "occupation_verbatim": null,
                    "occupation_normalised": null
                }
            ],

            "age": null
        }
    ],

    "buried": [],

    "deaths": []
}

For an actual burial/death age:

{
    "verbatim": "historical age expression",
    "age_days": null
}

For an actual traveller:

{
    "date": {
        "verbatim": null,
        "normalised": null
    },

    "gate": null,

    "direction": null,

    "person": {
        "given_name": null,
        "family_name": null,
        "occupation_verbatim": null,
        "occupation_normalised": null
    },

    "time": null,

    "transport": null,

    "origin": null,

    "lodging": null,

    "s_value": {
        "verbatim": null,
        "value": null
    }
}

Again:

The concrete values shown above are placeholders only.

Do NOT copy them into the answer.
"""


# PREPARE CHUNK

def prepare_chunk(
    current_issue_path,
    transcription_files
):

    chunk_texts = []

    issue_number = None
    issue_date = None
    canvases = []

    for filename in transcription_files:

        path = os.path.join(
            current_issue_path,
            filename
        )

        text = load_transcription(path)

        (
            current_issue_number,
            current_issue_date,
            canvas
        ) = get_transcription_metadata(text)

        if issue_number is None:
            issue_number = current_issue_number

        if issue_date is None:
            issue_date = current_issue_date

        if canvas is not None:
            canvases.append(canvas)

        # Remove metadata header before sending to VLM

        marker = "--- TRANSCRIPTION ---"

        if marker in text:

            text = text.split(
                marker,
                1
            )[1]

        # Add canvas separator

        chunk_texts.append(
            f"""
============================================================
CANVAS {canvas}
============================================================

{text.strip()}
"""
        )

    combined_text = "\n".join(
        chunk_texts
    )

    return (
        combined_text,
        issue_number,
        issue_date,
        canvases
    )


# VLM EXTRACTION

def extract_chunk_with_vlm(
    chunk_text
):

    prompt = (
        EXTRACTION_PROMPT
        + "\n\n"
        + "============================================================\n"
        + "TRANSCRIPTION TO PROCESS\n"
        + "============================================================\n\n"
        + chunk_text
    )

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": prompt
                }
            ]
        }
    ]

    # Chat template

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Tokenize

    inputs = processor(
        text=[text],
        return_tensors="pt"
    )

    inputs = {
        key: value.to(model.device)
        if hasattr(value, "to")
        else value
        for key, value in inputs.items()
    }

    # Generate

    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_new_tokens=4096,
            do_sample=False
        )

    # Remove prompt tokens

    input_length = (
        inputs["input_ids"].shape[1]
    )

    generated_ids = (
        output_ids[:, input_length:]
    )

    output = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    # GPU cleanup

    del inputs
    del output_ids
    del generated_ids

    torch.cuda.empty_cache()
    gc.collect()

    return output



# JSON PARSING

def parse_json_output(output):

    """
    Parse the VLM output.

    No schema validation.
    No normalization.
    No entity cleanup.
    No content modification.

    json_repair is only used if the VLM produced invalid JSON.
    Steps above produced inconsistent results during testing and destroyed information
    """

    output = output.strip()

    # Remove markdown fences if present

    if output.startswith("```"):

        output = re.sub(
            r"^```(?:json)?\s*",
            "",
            output,
            flags=re.IGNORECASE
        )

        output = re.sub(
            r"\s*```$",
            "",
            output
        )

        output = output.strip()

    # Try normal JSON

    try:

        return json.loads(
            output
        )

    except json.JSONDecodeError:

        pass

    # Try to isolate JSON object

    start = output.find("{")
    end = output.rfind("}")

    if (
        start != -1
        and end != -1
        and end > start
    ):

        candidate = output[
            start:end + 1
        ]

        try:

            return json.loads(
                candidate
            )

        except json.JSONDecodeError:

            pass

    # JSON repair

    repaired = repair_json(
        output
    )

    return json.loads(
        repaired
    )



# ADD SCRIPT-GENERATED METADATA

def add_source_metadata(
    extracted_data,
    issue_number,
    issue_date,
    canvases
):

    """
    Add the four metadata values from the actual
    transcription files.

    The VLM does NOT provide these values.
    """

    if not isinstance(
        extracted_data,
        dict
    ):

        raise ValueError(
            "VLM output is not a JSON object."
        )


    # ID

    extracted_data["id"] = (
        f"{issue_date}_Num_{issue_number}"
    )

    # Source metadata

    extracted_data["source"] = {
        "issue_date": issue_date,
        "issue_number": f"Num. {issue_number}",
        "canvases": canvases
    }

    return extracted_data



# MERGE CHUNKS

def merge_values(
    first,
    second
):

    """
    Generic merge.

    No normalization.
    No validation.
    No entity manipulation.
    No deduplication.

    Dictionaries are recursively merged.
    Lists are simply concatenated.
    """

    if (
        isinstance(first, dict)
        and isinstance(second, dict)
    ):

        merged = dict(first)

        for key, value in second.items():

            if key not in merged:

                merged[key] = value

            else:

                merged[key] = merge_values(
                    merged[key],
                    value
                )

        return merged

    if (
        isinstance(first, list)
        and isinstance(second, list)
    ):

        return first + second


    return first


def merge_chunk_results(
    chunk_results
):

    if not chunk_results:

        raise ValueError(
            "No chunk results to merge."
        )

    merged = chunk_results[0]

    for chunk in chunk_results[1:]:

        merged = merge_values(
            merged,
            chunk
        )

    return merged



# CREATE ENTITY BASE FOLDER

# Only create the base output folder once we know that processing is going to take place.

if not os.path.exists(entity_base):

    print(
        "Entity output folder does not exist."
    )

    print(
        f"Creating: {entity_base}"
    )

    os.makedirs(
        entity_base,
        exist_ok=True
    )

else:

    print(
        f"Entity output folder already exists: {entity_base}"
    )


# FIND ALL ISSUE FOLDERS

issue_folders = [
    folder
    for folder in os.listdir(
        transcription_base
    )
    if os.path.isdir(
        os.path.join(
            transcription_base,
            folder
        )
    )
]


# Sort numerically

issue_folders = sorted(
    issue_folders,
    key=get_issue_number
)


print(
    f"\nFound {len(issue_folders)} issue folders "
    f"in RegDia_transcriptions."
)



# CHECK EXISTING ENTITY OUTPUTS FIRST

print("\n")
print("=" * 70)
print("CHECKING EXISTING ENTITY JSON FILES")
print("=" * 70)


completed_issues = []
issues_to_process = []


for issue_folder in issue_folders:

    final_json_path = os.path.join(
        entity_base,
        issue_folder,
        f"{issue_folder}.json"
    )

    if os.path.isfile(
        final_json_path
    ):

        completed_issues.append(
            issue_folder
        )

    else:

        issues_to_process.append(
            issue_folder
        )


print(
    f"Completed issues already found: "
    f"{len(completed_issues)}"
)

print(
    f"Issues still to process: "
    f"{len(issues_to_process)}"
)


if completed_issues:

    print("\nAlready completed:")

    for issue in completed_issues:

        print(
            f"  ✓ {issue}"
        )


if issues_to_process:

    print("\nTo process:")

    for issue in issues_to_process:

        print(
            f"  → {issue}"
        )


# PROCESS ONLY INCOMPLETE ISSUES

for issue_folder in issues_to_process:

    print("\n")
    print("=" * 70)
    print(
        f"PROCESSING ISSUE: {issue_folder}"
    )
    print("=" * 70)

    current_issue_path = os.path.join(
        transcription_base,
        issue_folder
    )

    issue_output_path = os.path.join(
        entity_base,
        issue_folder
    )

    os.makedirs(
        issue_output_path,
        exist_ok=True
    )

    final_json_path = os.path.join(
        issue_output_path,
        f"{issue_folder}.json"
    )


    if os.path.isfile(
        final_json_path
    ):

        print(
            "↪ Final JSON exists."
        )

        print(
            "  Skipping issue."
        )

        continue


    # Find transcription files

    transcription_files = [
        filename
        for filename in os.listdir(
            current_issue_path
        )
        if filename.lower().endswith(".txt")
    ]

    transcription_files = sorted(
        transcription_files,
        key=get_canvas_number
    )

    if not transcription_files:

        print(
            "⚠ No transcription files found."
        )

        continue

    print(
        f"Pages: {len(transcription_files)}"
    )


    # Get issue metadata DIRECTLY from transcription files

    (
        actual_issue_number,
        actual_issue_date,
        _
    ) = get_transcription_metadata(
        load_transcription(
            os.path.join(
                current_issue_path,
                transcription_files[0]
            )
        )
    )

    all_canvases = [
        get_canvas_number(filename)
        for filename in transcription_files
    ]

    print(
        f"Issue number: {actual_issue_number}"
    )

    print(
        f"Issue date:   {actual_issue_date}"
    )

    print(
        f"Canvases:     {all_canvases}"
    )


    # Create chunks

    chunks = [
        transcription_files[
            i:i + CHUNK_SIZE
        ]
        for i in range(
            0,
            len(transcription_files),
            CHUNK_SIZE
        )
    ]

    print(
        f"Chunks: {len(chunks)} "
        f"(up to {CHUNK_SIZE} pages each)"
    )

    chunk_results = []

    all_chunks_succeeded = True

    # PROCESS CHUNKS

    for chunk_index, chunk_files in enumerate(
        chunks,
        start=1
    ):

        print("\n")
        print(
            f"--- Chunk {chunk_index}/{len(chunks)} ---"
        )

        chunk_canvases = [
            get_canvas_number(filename)
            for filename in chunk_files
        ]

        print(
            "Canvases:",
            chunk_canvases
        )

        try:


            # Prepare transcription

            (
                chunk_text,
                chunk_issue_number,
                chunk_issue_date,
                canvases
            ) = prepare_chunk(
                current_issue_path,
                chunk_files
            )


            # VLM

            raw_output = extract_chunk_with_vlm(
                chunk_text
            )


            extracted_data = parse_json_output(
                raw_output
            )

            # Add metadata ONLY HERE

            extracted_data = add_source_metadata(
                extracted_data,
                actual_issue_number,
                actual_issue_date,
                canvases
            )

            # Save chunk

            chunk_filename = (
                f"{issue_folder}"
                f"_chunk_{chunk_index}.json"
            )

            chunk_path = os.path.join(
                issue_output_path,
                chunk_filename
            )

            with open(
                chunk_path,
                "w",
                encoding="utf-8"
            ) as f:

                json.dump(
                    extracted_data,
                    f,
                    ensure_ascii=False,
                    indent=4
                )

            print(
                f"✓ Saved {chunk_filename}"
            )

            chunk_results.append(
                extracted_data
            )

        except Exception as e:

            print(
                f"\n✗ ERROR in chunk "
                f"{chunk_index}:"
            )

            print(
                e
            )

            all_chunks_succeeded = False

            continue


    # DO NOT CREATE FINAL JSON IF A CHUNK FAILED

    if not all_chunks_succeeded:

        print("\n")
        print(
            "⚠ Not all chunks succeeded."
        )

        print(
            "Final JSON will NOT be created."
        )

        print(
            "The issue will be retried on the next run."
        )

        continue

    if len(chunk_results) != len(chunks):

        print("\n")
        print(
            "⚠ Number of successful chunks does not "
            "match expected number."
        )

        print(
            "Final JSON will NOT be created."
        )

        continue

    # MERGE

    print("\n")
    print(
        f"Merging {len(chunk_results)} chunk(s)..."
    )

    final_result = merge_chunk_results(
        chunk_results
    )


    # OVERWRITE METADATA FROM ACTUAL SOURCE

    final_result["id"] = (
        f"{actual_issue_date}_Num_{actual_issue_number}"
    )

    final_result["source"] = {
        "issue_date": actual_issue_date,
        "issue_number": f"Num. {actual_issue_number}",
        "canvases": all_canvases
    }

    # SAVE FINAL JSON

    with open(
        final_json_path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            final_result,
            f,
            ensure_ascii=False,
            indent=4
        )

    print(
        "\n✓ FINAL JSON SAVED:"
    )

    print(
        final_json_path
    )


    # Memory cleanup

    gc.collect()
    torch.cuda.empty_cache()



# FINISHED

print("\n")
print("=" * 70)
print("PROCESSING FINISHED")
print("=" * 70)

print(
    f"\nAlready completed: {len(completed_issues)}"
)

print(
    f"Processed this run: {len(issues_to_process)}"
)

Entity output folder already exists: /content/drive/MyDrive/Colab Notebooks/RegDia_entites

Found 52 issue folders in RegDia_transcriptions.


CHECKING EXISTING ENTITY JSON FILES
Completed issues already found: 51
Issues still to process: 1

Already completed:
  ✓ Num_1_02_01_1770
  ✓ Num_2_09_01_1770
  ✓ Num_3_16_01_1770
  ✓ Num_4_23_01_1770
  ✓ Num_5_30_01_1770
  ✓ Num_6_06_02_1770
  ✓ Num_7_13_02_1770
  ✓ Num_8_20_02_1770
  ✓ Num_9_27_02_1770
  ✓ Num_10_06_03_1770
  ✓ Num_11_13_03_1770
  ✓ Num_12_20_03_1770
  ✓ Num_13_27_03_1770
  ✓ Num_14_03_04_1770
  ✓ Num_15_10_04_1770
  ✓ Num_16_17_04_1770
  ✓ Num_17_24_04_1770
  ✓ Num_18_01_05_1770
  ✓ Num_19_08_05_1770
  ✓ Num_20_15_05_1770
  ✓ Num_21_22_05_1770
  ✓ Num_22_29_05_1770
  ✓ Num_23_05_06_1770
  ✓ Num_24_12_06_1770
  ✓ Num_25_19_06_1770
  ✓ Num_26_26_06_1770
  ✓ Num_27_03_07_1770
  ✓ Num_28_10_07_1770
  ✓ Num_29_17_07_1770
  ✓ Num_30_24_07_1770
  ✓ Num_31_31_07_1770
  ✓ Num_32_07_08_1770
  ✓ Num_33_14_08_1770
  ✓ Num_34_21_08_1770


In [7]:
# EVALUATION

import os
import json
import re
import csv
from collections import Counter, defaultdict


# CONFIGURATION

BASE = "/content/drive/MyDrive/Colab Notebooks"

ENTITIES_BASE = os.path.join(BASE, "RegDia_entities")
HAND_BASE = os.path.join(BASE, "RegDia_entities_hand_extracted")
REPAIRED_BASE = os.path.join(BASE, "RegDia_entities_repaired")

# Reports will be written here
REPORT_BASE = os.path.join(BASE, "RegDia_evaluation")
os.makedirs(REPORT_BASE, exist_ok=True)


# HELPERS

def load_json(path):
    """Load JSON and return (data, error)."""
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f), None
    except Exception as e:
        return None, str(e)


def get_issue_id(filename):
    """
    Convert filenames into a common issue ID.

    Examples:
        Num_2_09_01_1770.json
        -> Num_2_09_01_1770

        Num_2_09_01_1770_hand_extracted.json
        -> Num_2_09_01_1770
    """
    name = os.path.basename(filename)

    name = re.sub(r"_hand_extracted(?=\.json$)", "", name)
    name = re.sub(r"\.json$", "", name)

    return name


def find_vlm_files():
    """
    Find all VLM JSON files.

    Expected structure:
        RegDia_entities/
            Num_x_xx_xx_xxxx/
                Num_x_xx_xx_xxxx.json
    """
    files = {}

    if not os.path.exists(ENTITIES_BASE):
        print(f"ERROR: Folder does not exist:\n{ENTITIES_BASE}")
        return files

    for folder_name in sorted(os.listdir(ENTITIES_BASE)):
        folder_path = os.path.join(ENTITIES_BASE, folder_name)

        if not os.path.isdir(folder_path):
            continue

        json_name = f"{folder_name}.json"
        json_path = os.path.join(folder_path, json_name)

        if os.path.isfile(json_path):
            issue_id = get_issue_id(json_name)
            files[issue_id] = json_path

    return files


def find_hand_files():
    """
    Find hand-extracted JSON files.

    Expected structure:
        RegDia_entities_hand_extracted/
            Num_2_09_01_1770_hand_extracted.json
    """
    files = {}

    if not os.path.exists(HAND_BASE):
        print(f"WARNING: Hand-extracted folder does not exist:\n{HAND_BASE}")
        return files

    for filename in sorted(os.listdir(HAND_BASE)):
        if not filename.endswith(".json"):
            continue

        path = os.path.join(HAND_BASE, filename)

        if not os.path.isfile(path):
            continue

        issue_id = get_issue_id(filename)
        files[issue_id] = path

    return files


def find_repaired_files():
    """
    Find repaired JSON files if they already exist.

    This is only reported for now.
    Nothing is modified.
    """
    files = {}

    if not os.path.exists(REPAIRED_BASE):
        return files

    for filename in sorted(os.listdir(REPAIRED_BASE)):
        # Support both:
        #   folder/file.json
        # and potentially flat files.
        if filename.endswith(".json"):
            path = os.path.join(REPAIRED_BASE, filename)

            if os.path.isfile(path):
                issue_id = get_issue_id(filename)
                files[issue_id] = path

        elif os.path.isdir(os.path.join(REPAIRED_BASE, filename)):
            folder_path = os.path.join(REPAIRED_BASE, filename)
            json_name = f"{filename}.json"
            json_path = os.path.join(folder_path, json_name)

            if os.path.isfile(json_path):
                issue_id = get_issue_id(json_name)
                files[issue_id] = json_path

    return files


# STRUCTURAL INSPECTION

EXPECTED_TOP_LEVEL_KEYS = {
    "evangelisch",
    "catholisch",
    "arrivals_departures",
}


def inspect_json(path):
    """
    Inspect one VLM JSON.

    This does NOT evaluate semantic correctness.
    It only checks structural properties and counts.
    """

    result = {
        "valid_json": False,
        "json_error": None,

        "is_dict": False,

        "has_id": False,
        "has_source": False,

        "top_level_keys": [],
        "missing_top_level_keys": [],
        "unexpected_top_level_keys": [],

        "source_issue_date": None,
        "source_issue_number": None,
        "source_canvases_count": None,

        "evangelisch_present": False,
        "catholisch_present": False,
        "arrivals_departures_present": False,

        "parish_count_evangelisch": 0,
        "parish_count_catholisch": 0,

        "baptised_count": 0,
        "buried_count": 0,
        "deaths_count": 0,
        "arrivals_departures_count": 0,

        "person_count": 0,

        "empty_person_records": 0,
        "duplicate_json_keys": None,
    }

    data, error = load_json(path)

    if error is not None:
        result["json_error"] = error
        return result

    result["valid_json"] = True

    if not isinstance(data, dict):
        return result

    result["is_dict"] = True


    # Top-level structure

    result["top_level_keys"] = sorted(data.keys())

    result["has_id"] = "id" in data
    result["has_source"] = "source" in data

    actual_content_keys = set(data.keys()) - {"id", "source"}

    result["missing_top_level_keys"] = sorted(
        EXPECTED_TOP_LEVEL_KEYS - actual_content_keys
    )

    result["unexpected_top_level_keys"] = sorted(
        actual_content_keys - EXPECTED_TOP_LEVEL_KEYS
    )

    result["evangelisch_present"] = "evangelisch" in data
    result["catholisch_present"] = "catholisch" in data
    result["arrivals_departures_present"] = "arrivals_departures" in data


    # Source metadata

    source = data.get("source")

    if isinstance(source, dict):
        result["source_issue_date"] = source.get("issue_date")
        result["source_issue_number"] = source.get("issue_number")

        canvases = source.get("canvases")

        if isinstance(canvases, list):
            result["source_canvases_count"] = len(canvases)


    # Evangelisch / Catholisch

    for religion_key, parish_counter_key in [
        ("evangelisch", "parish_count_evangelisch"),
        ("catholisch", "parish_count_catholisch"),
    ]:

        section = data.get(religion_key)

        if not isinstance(section, dict):
            continue

        parishes = section.get("parishes", [])

        if not isinstance(parishes, list):
            continue

        result[parish_counter_key] = len(parishes)

        for parish in parishes:

            if not isinstance(parish, dict):
                continue


            # Baptised

            baptised = parish.get("baptised", [])

            if isinstance(baptised, list):
                result["baptised_count"] += len(baptised)

                for record in baptised:
                    if not isinstance(record, dict):
                        continue

                    persons = record.get("persons", [])

                    if isinstance(persons, list):
                        result["person_count"] += len(persons)

                        for person in persons:
                            if isinstance(person, dict):
                                if not person:
                                    result["empty_person_records"] += 1


            # Buried

            buried = parish.get("buried", [])

            if isinstance(buried, list):
                result["buried_count"] += len(buried)

                for record in buried:
                    if not isinstance(record, dict):
                        continue

                    persons = record.get("persons", [])

                    if isinstance(persons, list):
                        result["person_count"] += len(persons)

                        for person in persons:
                            if isinstance(person, dict):
                                if not person:
                                    result["empty_person_records"] += 1

            # Deaths

            deaths = parish.get("deaths", [])

            if isinstance(deaths, list):
                result["deaths_count"] += len(deaths)

                for record in deaths:
                    if not isinstance(record, dict):
                        continue

                    persons = record.get("persons", [])

                    if isinstance(persons, list):
                        result["person_count"] += len(persons)

                        for person in persons:
                            if isinstance(person, dict):
                                if not person:
                                    result["empty_person_records"] += 1


    # Arrivals / departures

    arrivals = data.get("arrivals_departures", [])

    if isinstance(arrivals, list):
        result["arrivals_departures_count"] = len(arrivals)

        for record in arrivals:

            if not isinstance(record, dict):
                continue

            person = record.get("person")

            if isinstance(person, dict):
                result["person_count"] += 1

                if not person:
                    result["empty_person_records"] += 1

    return result



# FIND FILES

vlm_files = find_vlm_files()
hand_files = find_hand_files()
repaired_files = find_repaired_files()


print("=" * 70)
print("REGENSBURGER DIARIUM – CORPUS INSPECTION")
print("=" * 70)

print(f"\nVLM JSON files found:          {len(vlm_files)}")
print(f"Hand-extracted JSONs found:    {len(hand_files)}")
print(f"Repaired JSONs found:          {len(repaired_files)}")


# MATCH HAND-EXTRACTED FILES

matched_hand = []
unmatched_hand = []
missing_hand_counterpart = []

for issue_id, hand_path in sorted(hand_files.items()):

    if issue_id in vlm_files:
        matched_hand.append(issue_id)
    else:
        unmatched_hand.append(issue_id)


for issue_id in sorted(vlm_files):

    if issue_id not in hand_files:
        missing_hand_counterpart.append(issue_id)


print("\n" + "-" * 70)
print("HAND-EXTRACTED MATCHING")
print("-" * 70)

print(f"Matched hand-extracted issues: {len(matched_hand)}")

if unmatched_hand:
    print("\nHand-extracted files without matching VLM JSON:")
    for issue_id in unmatched_hand:
        print(f"  - {issue_id}")

if matched_hand:
    print("\nMatched issues:")
    for issue_id in matched_hand:
        print(f"  ✓ {issue_id}")



# INSPECT ALL VLM JSONS

reports = {}

for issue_id, path in sorted(vlm_files.items()):
    reports[issue_id] = inspect_json(path)



# CORPUS-LEVEL SUMMARY

total = len(reports)

valid_json = sum(r["valid_json"] for r in reports.values())
valid_dict = sum(r["is_dict"] for r in reports.values())

all_expected_sections = sum(
    len(r["missing_top_level_keys"]) == 0
    for r in reports.values()
)

with_evangelisch = sum(
    r["evangelisch_present"]
    for r in reports.values()
)

with_catholisch = sum(
    r["catholisch_present"]
    for r in reports.values()
)

with_arrivals = sum(
    r["arrivals_departures_present"]
    for r in reports.values()
)

with_source = sum(
    r["has_source"]
    for r in reports.values()
)

with_id = sum(
    r["has_id"]
    for r in reports.values()
)


print("\n" + "=" * 70)
print("STRUCTURAL SUMMARY")
print("=" * 70)

print(f"""
Total VLM JSONs:                       {total}

Valid JSON:                            {valid_json}/{total}
JSON root is object:                   {valid_dict}/{total}

Has expected top-level sections:       {all_expected_sections}/{total}

Has 'evangelisch':                     {with_evangelisch}/{total}
Has 'catholisch':                      {with_catholisch}/{total}
Has 'arrivals_departures':             {with_arrivals}/{total}

Has 'id':                              {with_id}/{total}
Has 'source':                          {with_source}/{total}
""")


# TOTAL RECORD COUNTS

total_parishes_ev = sum(
    r["parish_count_evangelisch"]
    for r in reports.values()
)

total_parishes_ca = sum(
    r["parish_count_catholisch"]
    for r in reports.values()
)

total_baptised = sum(
    r["baptised_count"]
    for r in reports.values()
)

total_buried = sum(
    r["buried_count"]
    for r in reports.values()
)

total_deaths = sum(
    r["deaths_count"]
    for r in reports.values()
)

total_arrivals = sum(
    r["arrivals_departures_count"]
    for r in reports.values()
)

total_persons = sum(
    r["person_count"]
    for r in reports.values()
)

total_empty_persons = sum(
    r["empty_person_records"]
    for r in reports.values()
)


print("\n" + "=" * 70)
print("EXTRACTED RECORD COUNTS")
print("=" * 70)

print(f"""
Evangelical parishes:                  {total_parishes_ev}
Catholic parishes:                     {total_parishes_ca}

Baptised records:                      {total_baptised}
Buried records:                        {total_buried}
Death records:                         {total_deaths}

Arrivals/departures:                   {total_arrivals}

Person objects:                        {total_persons}
Empty person objects:                  {total_empty_persons}
""")



# ISSUES WITH MISSING TOP-LEVEL SECTIONS

print("\n" + "=" * 70)
print("MISSING TOP-LEVEL SECTIONS")
print("=" * 70)

issues_with_missing_sections = []

for issue_id, report in sorted(reports.items()):

    if report["missing_top_level_keys"]:

        issues_with_missing_sections.append(issue_id)

        print(
            f"{issue_id}: "
            + ", ".join(report["missing_top_level_keys"])
        )

if not issues_with_missing_sections:
    print("None.")



# ISSUE-BY-ISSUE TABLE

print("\n" + "=" * 70)
print("ISSUE-BY-ISSUE SUMMARY")
print("=" * 70)

header = (
    f"{'Issue':<30}"
    f"{'EV':>4}"
    f"{'CA':>4}"
    f"{'Bap':>6}"
    f"{'Bur':>6}"
    f"{'Death':>7}"
    f"{'Arr':>6}"
    f"{'Pers':>7}"
)

print(header)
print("-" * len(header))

for issue_id, r in sorted(reports.items()):

    print(
        f"{issue_id:<30}"
        f"{r['parish_count_evangelisch']:>4}"
        f"{r['parish_count_catholisch']:>4}"
        f"{r['baptised_count']:>6}"
        f"{r['buried_count']:>6}"
        f"{r['deaths_count']:>7}"
        f"{r['arrivals_departures_count']:>6}"
        f"{r['person_count']:>7}"
    )



# SAVE DETAILED REPORT

report_json_path = os.path.join(
    REPORT_BASE,
    "corpus_structure_report.json"
)

with open(report_json_path, "w", encoding="utf-8") as f:
    json.dump(
        reports,
        f,
        ensure_ascii=False,
        indent=2
    )



# SAVE CSV REPORT

report_csv_path = os.path.join(
    REPORT_BASE,
    "corpus_structure_report.csv"
)

csv_fields = [
    "issue_id",
    "valid_json",
    "json_error",
    "is_dict",
    "has_id",
    "has_source",
    "top_level_keys",
    "missing_top_level_keys",
    "unexpected_top_level_keys",
    "source_issue_date",
    "source_issue_number",
    "source_canvases_count",
    "evangelisch_present",
    "catholisch_present",
    "arrivals_departures_present",
    "parish_count_evangelisch",
    "parish_count_catholisch",
    "baptised_count",
    "buried_count",
    "deaths_count",
    "arrivals_departures_count",
    "person_count",
    "empty_person_records",
    "duplicate_json_keys",
    "has_hand_extracted",
    "has_repaired",
]

with open(report_csv_path, "w", encoding="utf-8", newline="") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=csv_fields
    )

    writer.writeheader()

    for issue_id, report in sorted(reports.items()):

        row = {
            "issue_id": issue_id,
            **report,
            "top_level_keys": "|".join(
                report["top_level_keys"]
            ),
            "missing_top_level_keys": "|".join(
                report["missing_top_level_keys"]
            ),
            "unexpected_top_level_keys": "|".join(
                report["unexpected_top_level_keys"]
            ),
            "has_hand_extracted": issue_id in hand_files,
            "has_repaired": issue_id in repaired_files,
        }

        writer.writerow(row)



# FINAL OUTPUT

print("\n" + "=" * 70)
print("REPORTS SAVED")
print("=" * 70)

print(f"\nJSON report:")
print(report_json_path)

print(f"\nCSV report:")
print(report_csv_path)

print("\nNo source JSON files were modified.")


REGENSBURGER DIARIUM – CORPUS INSPECTION

VLM JSON files found:          52
Hand-extracted JSONs found:    3
Repaired JSONs found:          0

----------------------------------------------------------------------
HAND-EXTRACTED MATCHING
----------------------------------------------------------------------
Matched hand-extracted issues: 3

Matched issues:
  ✓ Num_2_09_01_1770
  ✓ Num_31_31_07_1770
  ✓ Num_3_16_01_1770

STRUCTURAL SUMMARY

Total VLM JSONs:                       52

Valid JSON:                            52/52
JSON root is object:                   52/52

Has expected top-level sections:       52/52

Has 'evangelisch':                     52/52
Has 'catholisch':                      52/52
Has 'arrivals_departures':             52/52

Has 'id':                              52/52
Has 'source':                          52/52


EXTRACTED RECORD COUNTS

Evangelical parishes:                  83
Catholic parishes:                     122

Baptised records:                    

In [8]:
import os
import json
import re
import csv
import unicodedata
from collections import defaultdict
from difflib import SequenceMatcher


# CONFIGURATION

BASE = "/content/drive/MyDrive/Colab Notebooks"

ENTITIES_BASE = os.path.join(
    BASE,
    "RegDia_entities"
)

HAND_BASE = os.path.join(
    BASE,
    "RegDia_entities_hand_extracted"
)

REPORT_BASE = os.path.join(
    BASE,
    "RegDia_evaluation"
)

os.makedirs(
    REPORT_BASE,
    exist_ok=True
)



# SCHEMA

CHURCH_EVENT_TYPES = [
    "baptised",
    "buried",
    "deaths"
]

ALL_EVENT_TYPES = [
    "baptised",
    "buried",
    "deaths",
    "arrivals_departures"
]


# Event-level fields
CHURCH_FIELDS = [
    "date.verbatim",
    "date.normalised",
    "age.verbatim",
    "age.age_days"
]

ARRIVAL_FIELDS = [
    "date.verbatim",
    "date.normalised",
    "gate",
    "direction",
    "time",
    "transport",
    "origin",
    "lodging",
    "s_value.verbatim",
    "s_value.value"
]


# Person fields
PERSON_FIELDS = [
    "given_name",
    "family_name",
    "occupation_verbatim",
    "occupation_normalised"
]



# MATCHING CONFIGURATION

# Minimum event matching score.
#
# The score is deliberately based primarily on event identity:
#
#   date
#   person identity
#   direction
#
# It does NOT depend on fields such as gate, transport,
# occupation, lodging, etc.
#
# Those fields are evaluated only after an event has been
# aligned.
#
EVENT_MATCH_THRESHOLD_CHURCH = 5.0
EVENT_MATCH_THRESHOLD_ARRIVAL = 5.0


# Fuzzy similarity threshold for names.
#
# 1.0 = identical
# 0.0 = completely different
#
NAME_FUZZY_THRESHOLD = 0.70



# FILE HELPERS

def load_json(path):

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        return json.load(f)


def get_issue_id(filename):

    name = os.path.basename(
        filename
    )

    name = re.sub(
        r"_hand_extracted(?=\.json$)",
        "",
        name
    )

    name = re.sub(
        r"\.json$",
        "",
        name
    )

    return name


def find_vlm_files():

    files = {}

    if not os.path.isdir(
        ENTITIES_BASE
    ):

        raise FileNotFoundError(
            f"VLM folder not found:\n{ENTITIES_BASE}"
        )

    for folder_name in sorted(
        os.listdir(ENTITIES_BASE)
    ):

        folder_path = os.path.join(
            ENTITIES_BASE,
            folder_name
        )

        if not os.path.isdir(
            folder_path
        ):
            continue

        filename = (
            f"{folder_name}.json"
        )

        path = os.path.join(
            folder_path,
            filename
        )

        if os.path.isfile(path):

            files[
                folder_name
            ] = path

    return files


def find_hand_files():

    files = {}

    if not os.path.isdir(
        HAND_BASE
    ):

        raise FileNotFoundError(
            f"Hand-extracted folder not found:\n{HAND_BASE}"
        )

    for filename in sorted(
        os.listdir(HAND_BASE)
    ):

        if not filename.endswith(
            ".json"
        ):
            continue

        path = os.path.join(
            HAND_BASE,
            filename
        )

        if not os.path.isfile(
            path
        ):
            continue

        issue_id = get_issue_id(
            filename
        )

        files[
            issue_id
        ] = path

    return files



# STRING NORMALIZATION

def normalize_string(value):

    if value is None:
        return ""

    if not isinstance(
        value,
        str
    ):

        value = str(value)

    value = unicodedata.normalize(
        "NFC",
        value
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value.casefold()


def normalize_name(value):

    """
    Slightly stronger normalization for person names.

    This is used ONLY for event/person matching.

    It does NOT change the actual field evaluation.
    """

    value = normalize_string(
        value
    )

    # Remove punctuation.
    value = re.sub(
        r"[^\w\s]",
        " ",
        value,
        flags=re.UNICODE
    )

    value = re.sub(
        r"\s+",
        " ",
        value
    ).strip()

    return value


def string_similarity(
    a,
    b
):

    a = normalize_name(a)
    b = normalize_name(b)

    if not a or not b:
        return 0.0

    if a == b:
        return 1.0

    return SequenceMatcher(
        None,
        a,
        b
    ).ratio()


def values_equal(
    a,
    b
):

    return (
        normalize_string(a)
        ==
        normalize_string(b)
    )



# NESTED VALUE ACCESS

def get_nested_value(
    data,
    path
):

    current = data

    for part in path.split("."):

        if not isinstance(
            current,
            dict
        ):
            return None

        current = current.get(
            part
        )

    return current



# PERSON HELPERS

def person_full_name(
    person
):

    if not isinstance(
        person,
        dict
    ):
        return ""

    given = person.get(
        "given_name"
    )

    family = person.get(
        "family_name"
    )

    parts = []

    if given:
        parts.append(
            str(given)
        )

    if family:
        parts.append(
            str(family)
        )

    return " ".join(
        parts
    )


def person_name_key(
    person
):

    return normalize_name(
        person_full_name(
            person
        )
    )


def person_name_similarity(
    gt_person,
    pred_person
):

    gt_full = person_full_name(
        gt_person
    )

    pred_full = person_full_name(
        pred_person
    )

    full_similarity = (
        string_similarity(
            gt_full,
            pred_full
        )
    )

    given_similarity = (
        string_similarity(
            gt_person.get(
                "given_name"
            ),
            pred_person.get(
                "given_name"
            )
        )
    )

    family_similarity = (
        string_similarity(
            gt_person.get(
                "family_name"
            ),
            pred_person.get(
                "family_name"
            )
        )
    )

    similarities = [
        s
        for s in [
            full_similarity,
            given_similarity,
            family_similarity
        ]
        if s > 0
    ]

    if not similarities:
        return 0.0

    # Full name is strongest.
    return (
        0.50 * full_similarity
        + 0.25 * given_similarity
        + 0.25 * family_similarity
    )



# FLATTEN EVENTS

def extract_events(
    data
):

    events = []

    if not isinstance(
        data,
        dict
    ):
        return events



    # CHURCH EVENTS

    for religion in [
        "evangelisch",
        "catholisch"
    ]:

        religion_section = data.get(
            religion
        )

        if not isinstance(
            religion_section,
            dict
        ):
            continue

        parishes = religion_section.get(
            "parishes",
            []
        )

        if not isinstance(
            parishes,
            list
        ):
            continue

        for parish in parishes:

            if not isinstance(
                parish,
                dict
            ):
                continue

            parish_name = parish.get(
                "name"
            )

            for event_type in (
                CHURCH_EVENT_TYPES
            ):

                records = parish.get(
                    event_type,
                    []
                )

                if not isinstance(
                    records,
                    list
                ):
                    continue

                for record in records:

                    if not isinstance(
                        record,
                        dict
                    ):
                        continue

                    persons = record.get(
                        "persons",
                        []
                    )

                    if not isinstance(
                        persons,
                        list
                    ):

                        persons = []

                    events.append({

                        "event_type":
                            event_type,

                        "religion":
                            religion,

                        "parish":
                            parish_name,

                        "record":
                            record,

                        "persons":
                            persons
                    })



    # ARRIVALS / DEPARTURES

    records = data.get(
        "arrivals_departures",
        []
    )

    if isinstance(
        records,
        list
    ):

        for record in records:

            if not isinstance(
                record,
                dict
            ):
                continue

            person = record.get(
                "person"
            )

            persons = []

            if isinstance(
                person,
                dict
            ):

                persons.append(
                    person
                )

            events.append({

                "event_type":
                    "arrivals_departures",

                "religion":
                    None,

                "parish":
                    None,

                "record":
                    record,

                "persons":
                    persons
            })

    return events



# MATCHING HELPERS

def compare_optional(
    a,
    b
):

    a_norm = normalize_string(
        a
    )

    b_norm = normalize_string(
        b
    )

    if not a_norm or not b_norm:
        return 0

    if a_norm == b_norm:
        return 1

    return -1


def best_person_similarity(
    gt_persons,
    pred_persons
):

    if not gt_persons or not pred_persons:
        return 0.0

    best = 0.0

    for gt_person in gt_persons:

        for pred_person in pred_persons:

            similarity = (
                person_name_similarity(
                    gt_person,
                    pred_person
                )
            )

            best = max(
                best,
                similarity
            )

    return best


def best_person_pair(
    gt_persons,
    pred_persons
):

    best_score = 0.0
    best_gt = None
    best_pred = None

    if not gt_persons or not pred_persons:
        return (
            0.0,
            None,
            None
        )

    for gt_i, gt_person in enumerate(
        gt_persons
    ):

        for pred_i, pred_person in enumerate(
            pred_persons
        ):

            score = person_name_similarity(
                gt_person,
                pred_person
            )

            if score > best_score:

                best_score = score
                best_gt = gt_i
                best_pred = pred_i

    return (
        best_score,
        best_gt,
        best_pred
    )



# EVENT MATCH SCORE

def event_match_score(
    gt,
    pred
):

    """
    Calculate an event identity score.

    IMPORTANT:
    This does NOT evaluate extraction quality.

    It answers only:

        "Do these two records probably describe
         the same underlying historical event?"

    The score intentionally avoids fields such as:

        gate
        transport
        lodging
        occupation
        s_value

    because those fields are evaluated afterwards.
    """

    if (
        gt["event_type"]
        !=
        pred["event_type"]
    ):

        return (
            -999,
            {}
        )


    score = 0.0

    evidence = {}



    # CHURCH EVENTS

    if gt["event_type"] in CHURCH_EVENT_TYPES:

        # Religion
        if (
            gt.get("religion")
            ==
            pred.get("religion")
        ):

            score += 1.0
            evidence[
                "religion"
            ] = "match"

        else:

            evidence[
                "religion"
            ] = "different"


        # Parish
        parish_similarity = (
            string_similarity(
                gt.get("parish"),
                pred.get("parish")
            )
        )

        if parish_similarity >= 0.90:

            score += 2.0

        elif parish_similarity >= 0.70:

            score += 1.0

        evidence[
            "parish_similarity"
        ] = parish_similarity


        # Date
        gt_date = get_nested_value(
            gt["record"],
            "date.verbatim"
        )

        pred_date = get_nested_value(
            pred["record"],
            "date.verbatim"
        )

        date_result = compare_optional(
            gt_date,
            pred_date
        )

        evidence[
            "date"
        ] = date_result

        if date_result == 1:

            score += 4.0

        elif date_result == -1:

            # Different dates are evidence against a match.
            score -= 2.0


        # Person identity
        person_similarity = (
            best_person_similarity(
                gt.get("persons", []),
                pred.get("persons", [])
            )
        )

        evidence[
            "person_similarity"
        ] = person_similarity

        if person_similarity >= 0.90:

            score += 5.0

        elif person_similarity >= 0.80:

            score += 4.0

        elif person_similarity >= 0.70:

            score += 3.0

        elif person_similarity >= 0.55:

            score += 1.5

        else:

            score -= 2.0


        return (
            score,
            evidence
        )



    # ARRIVALS / DEPARTURES

    if gt["event_type"] == "arrivals_departures":

        # Date
        gt_date = get_nested_value(
            gt["record"],
            "date.verbatim"
        )

        pred_date = get_nested_value(
            pred["record"],
            "date.verbatim"
        )

        date_result = compare_optional(
            gt_date,
            pred_date
        )

        evidence[
            "date"
        ] = date_result

        if date_result == 1:

            score += 4.0

        elif date_result == -1:

            score -= 2.0


        # Direction
        gt_direction = gt[
            "record"
        ].get(
            "direction"
        )

        pred_direction = pred[
            "record"
        ].get(
            "direction"
        )

        direction_result = (
            compare_optional(
                gt_direction,
                pred_direction
            )
        )

        evidence[
            "direction"
        ] = direction_result

        if direction_result == 1:

            score += 3.0

        elif direction_result == -1:

            score -= 1.0


        # Person
        person_similarity = (
            best_person_similarity(
                gt.get("persons", []),
                pred.get("persons", [])
            )
        )

        evidence[
            "person_similarity"
        ] = person_similarity

        if person_similarity >= 0.90:

            score += 5.0

        elif person_similarity >= 0.80:

            score += 4.0

        elif person_similarity >= 0.70:

            score += 3.0

        elif person_similarity >= 0.55:

            score += 1.5

        else:

            score -= 2.0


        return (
            score,
            evidence
        )


    return (
        score,
        evidence
    )



# EVENT MATCHING

def match_events(
    ground_truth_events,
    predicted_events
):

    """
    One-to-one greedy event alignment.

    Candidates are ranked by identity score.

    Importantly, matching is performed separately by
    event type. A baptism can never match a burial, etc.
    """

    candidates = []

    for gt_index, gt in enumerate(
        ground_truth_events
    ):

        for pred_index, pred in enumerate(
            predicted_events
        ):

            score, evidence = (
                event_match_score(
                    gt,
                    pred
                )
            )

            if score < 0:
                continue

            if gt["event_type"] in CHURCH_EVENT_TYPES:

                threshold = (
                    EVENT_MATCH_THRESHOLD_CHURCH
                )

            else:

                threshold = (
                    EVENT_MATCH_THRESHOLD_ARRIVAL
                )

            if score >= threshold:

                candidates.append({

                    "score":
                        score,

                    "gt_index":
                        gt_index,

                    "pred_index":
                        pred_index,

                    "evidence":
                        evidence
                })


    # Strongest matches first
    candidates.sort(
        key=lambda x: x["score"],
        reverse=True
    )


    used_gt = set()
    used_pred = set()

    matches = []


    for candidate in candidates:

        gt_index = candidate[
            "gt_index"
        ]

        pred_index = candidate[
            "pred_index"
        ]

        if gt_index in used_gt:
            continue

        if pred_index in used_pred:
            continue

        used_gt.add(
            gt_index
        )

        used_pred.add(
            pred_index
        )

        matches.append({

            "gt_index":
                gt_index,

            "pred_index":
                pred_index,

            "match_score":
                candidate["score"],

            "evidence":
                candidate["evidence"]
        })


    unmatched_gt = [
        i
        for i in range(
            len(ground_truth_events)
        )
        if i not in used_gt
    ]

    unmatched_pred = [
        i
        for i in range(
            len(predicted_events)
        )
        if i not in used_pred
    ]


    return (
        matches,
        unmatched_gt,
        unmatched_pred
    )



# FIELD COMPARISON

def is_empty(
    value
):

    return (
        value is None
        or value == ""
    )


def compare_field(
    gt_value,
    pred_value
):

    """
    Returns:

        correct
        incorrect
        missing
        spurious
        both_null
    """

    gt_empty = is_empty(
        gt_value
    )

    pred_empty = is_empty(
        pred_value
    )

    if (
        gt_empty
        and
        pred_empty
    ):

        return "both_null"


    if (
        not gt_empty
        and
        pred_empty
    ):

        return "missing"


    if (
        gt_empty
        and
        not pred_empty
    ):

        return "spurious"


    if values_equal(
        gt_value,
        pred_value
    ):

        return "correct"


    return "incorrect"


# PERSON MATCHING

def person_match_score(
    gt_person,
    pred_person
):

    similarity = person_name_similarity(
        gt_person,
        pred_person
    )

    score = (
        similarity * 10
    )

    # Role provides additional evidence.
    role_result = compare_optional(
        gt_person.get("role"),
        pred_person.get("role")
    )

    if role_result == 1:

        score += 2

    elif role_result == -1:

        score -= 1

    return score


def match_persons(
    gt_persons,
    pred_persons
):

    """
    Match persons within an already aligned event.

    The matching itself is independent of the subsequent
    field evaluation.
    """

    candidates = []

    for gt_i, gt_person in enumerate(
        gt_persons
    ):

        for pred_i, pred_person in enumerate(
            pred_persons
        ):

            score = person_match_score(
                gt_person,
                pred_person
            )

            candidates.append(
                (
                    score,
                    gt_i,
                    pred_i
                )
            )


    candidates.sort(
        reverse=True
    )


    used_gt = set()
    used_pred = set()

    matches = []


    for score, gt_i, pred_i in candidates:

        if gt_i in used_gt:
            continue

        if pred_i in used_pred:
            continue

        # Require some identity evidence.
        if score < 4:

            continue

        used_gt.add(
            gt_i
        )

        used_pred.add(
            pred_i
        )

        matches.append({

            "gt_index":
                gt_i,

            "pred_index":
                pred_i,

            "match_score":
                score
        })


    # Safe fallback for one-to-one remaining persons.
    remaining_gt = [
        i
        for i in range(
            len(gt_persons)
        )
        if i not in used_gt
    ]

    remaining_pred = [
        i
        for i in range(
            len(pred_persons)
        )
        if i not in used_pred
    ]


    if (
        len(remaining_gt) == 1
        and
        len(remaining_pred) == 1
    ):

        matches.append({

            "gt_index":
                remaining_gt[0],

            "pred_index":
                remaining_pred[0],

            "match_score":
                0,

            "match_type":
                "one_to_one_fallback"
        })

        used_gt.add(
            remaining_gt[0]
        )

        used_pred.add(
            remaining_pred[0]
        )


    unmatched_gt = [
        i
        for i in range(
            len(gt_persons)
        )
        if i not in used_gt
    ]

    unmatched_pred = [
        i
        for i in range(
            len(pred_persons)
        )
        if i not in used_pred
    ]


    return (
        matches,
        unmatched_gt,
        unmatched_pred
    )



# APPLICABLE FIELDS

def applicable_event_fields(
    event_type
):

    if event_type in CHURCH_EVENT_TYPES:

        return CHURCH_FIELDS

    if event_type == "arrivals_departures":

        return ARRIVAL_FIELDS

    return []



# FIELD STATISTICS HELPERS

def new_field_stats():

    return {

        "correct": 0,
        "incorrect": 0,
        "missing": 0,
        "spurious": 0,
        "both_null": 0
    }


# EVALUATE ONE ISSUE

def evaluate_issue(
    issue_id,
    gt_data,
    pred_data
):

    gt_events = extract_events(
        gt_data
    )

    pred_events = extract_events(
        pred_data
    )



    # EVENT MATCHING

    (
        event_matches,
        unmatched_gt,
        unmatched_pred
    ) = match_events(
        gt_events,
        pred_events
    )


    tp = len(
        event_matches
    )

    fn = len(
        unmatched_gt
    )

    fp = len(
        unmatched_pred
    )


    precision = (
        tp / (tp + fp)
        if tp + fp
        else 0
    )

    recall = (
        tp / (tp + fn)
        if tp + fn
        else 0
    )

    f1 = (
        2 * precision * recall
        /
        (precision + recall)
        if precision + recall
        else 0
    )



    # EVENT TYPE METRICS

    event_metrics = {}


    for event_type in ALL_EVENT_TYPES:

        gt_count = sum(
            event["event_type"]
            == event_type
            for event in gt_events
        )

        pred_count = sum(
            event["event_type"]
            == event_type
            for event in pred_events
        )

        event_tp = sum(
            gt_events[
                match["gt_index"]
            ]["event_type"]
            == event_type
            for match in event_matches
        )

        event_fp = (
            pred_count
            - event_tp
        )

        event_fn = (
            gt_count
            - event_tp
        )

        p = (
            event_tp
            /
            (event_tp + event_fp)
            if event_tp + event_fp
            else 0
        )

        r = (
            event_tp
            /
            (event_tp + event_fn)
            if event_tp + event_fn
            else 0
        )

        f = (
            2 * p * r
            /
            (p + r)
            if p + r
            else 0
        )

        event_metrics[
            event_type
        ] = {

            "ground_truth":
                gt_count,

            "predicted":
                pred_count,

            "tp":
                event_tp,

            "fp":
                event_fp,

            "fn":
                event_fn,

            "precision":
                p,

            "recall":
                r,

            "f1":
                f
        }



    # FIELD STATISTICS
    #
    # conditional:
    # only aligned events are evaluated here

    field_stats = defaultdict(
        new_field_stats
    )


    # END-TO-END FIELD STATISTICS
    #
    # Every GT event contributes.
    #
    # If an event was not detected at all, all non-null GT
    # fields are counted as missing.
    #
    # This therefore measures end-to-end field recovery.

    field_end_to_end = defaultdict(
        new_field_stats
    )


    # ERROR ROWS

    error_rows = []

    # MATCHED EVENTS

    for match in event_matches:

        gt_event = gt_events[
            match["gt_index"]
        ]

        pred_event = pred_events[
            match["pred_index"]
        ]

        event_type = gt_event[
            "event_type"
        ]

        gt_record = gt_event[
            "record"
        ]

        pred_record = pred_event[
            "record"
        ]



        # EVENT-LEVEL FIELDS

        fields = applicable_event_fields(
            event_type
        )


        for field in fields:

            gt_value = get_nested_value(
                gt_record,
                field
            )

            pred_value = get_nested_value(
                pred_record,
                field
            )

            result = compare_field(
                gt_value,
                pred_value
            )


            field_stats[
                field
            ][result] += 1


            field_end_to_end[
                field
            ][result] += 1


            if result not in [
                "correct",
                "both_null"
            ]:

                error_rows.append({

                    "issue_id":
                        issue_id,

                    "event_type":
                        event_type,

                    "field":
                        field,

                    "ground_truth":
                        json.dumps(
                            gt_value,
                            ensure_ascii=False
                        ),

                    "prediction":
                        json.dumps(
                            pred_value,
                            ensure_ascii=False
                        ),

                    "error_type":
                        result,

                    "role":
                        "",

                    "event_match_score":
                        match["match_score"]
                })



        # PERSONS

        gt_persons = gt_event.get(
            "persons",
            []
        )

        pred_persons = pred_event.get(
            "persons",
            []
        )


        (
            person_matches,
            unmatched_gt_persons,
            unmatched_pred_persons
        ) = match_persons(
            gt_persons,
            pred_persons
        )


        # PERSON MATCHES

        for person_match in person_matches:

            gt_person = gt_persons[
                person_match["gt_index"]
            ]

            pred_person = pred_persons[
                person_match["pred_index"]
            ]

            role = gt_person.get(
                "role"
            )


            for field in PERSON_FIELDS:

                gt_value = gt_person.get(
                    field
                )

                pred_value = pred_person.get(
                    field
                )

                result = compare_field(
                    gt_value,
                    pred_value
                )


                field_name = (
                    f"person.{field}"
                )


                field_stats[
                    field_name
                ][result] += 1


                field_end_to_end[
                    field_name
                ][result] += 1


                if result not in [
                    "correct",
                    "both_null"
                ]:

                    error_rows.append({

                        "issue_id":
                            issue_id,

                        "event_type":
                            event_type,

                        "field":
                            field_name,

                        "ground_truth":
                            json.dumps(
                                gt_value,
                                ensure_ascii=False
                            ),

                        "prediction":
                            json.dumps(
                                pred_value,
                                ensure_ascii=False
                            ),

                        "error_type":
                            result,

                        "role":
                            role,

                        "event_match_score":
                            match["match_score"]
                    })



        # UNMATCHED GT PERSONS

        for gt_i in unmatched_gt_persons:

            gt_person = gt_persons[
                gt_i
            ]

            role = gt_person.get(
                "role"
            )


            error_rows.append({

                "issue_id":
                    issue_id,

                "event_type":
                    event_type,

                "field":
                    "person",

                "ground_truth":
                    json.dumps(
                        gt_person,
                        ensure_ascii=False
                    ),

                "prediction":
                    "",

                "error_type":
                    "missing_person",

                "role":
                    role,

                "event_match_score":
                    match["match_score"]
            })


            # Every non-null field in an unmatched GT
            # person is end-to-end missing.

            for field in PERSON_FIELDS:

                gt_value = gt_person.get(
                    field
                )

                if not is_empty(
                    gt_value
                ):

                    field_name = (
                        f"person.{field}"
                    )

                    field_end_to_end[
                        field_name
                    ]["missing"] += 1



        # UNMATCHED PREDICTED PERSONS

        for pred_i in unmatched_pred_persons:

            pred_person = pred_persons[
                pred_i
            ]

            error_rows.append({

                "issue_id":
                    issue_id,

                "event_type":
                    event_type,

                "field":
                    "person",

                "ground_truth":
                    "",

                "prediction":
                    json.dumps(
                        pred_person,
                        ensure_ascii=False
                    ),

                "error_type":
                    "spurious_person",

                "role":
                    pred_person.get(
                        "role",
                        ""
                    ),

                "event_match_score":
                    match["match_score"]
            })



    # UNMATCHED GT EVENTS
    #
    # These events contribute missing values to the
    # end-to-end field evaluation.

    for gt_i in unmatched_gt:

        gt_event = gt_events[
            gt_i
        ]

        event_type = gt_event[
            "event_type"
        ]

        gt_record = gt_event[
            "record"
        ]


        # Event-level fields

        for field in applicable_event_fields(
            event_type
        ):

            gt_value = get_nested_value(
                gt_record,
                field
            )

            if not is_empty(
                gt_value
            ):

                field_end_to_end[
                    field
                ]["missing"] += 1


        # Persons

        gt_persons = gt_event.get(
            "persons",
            []
        )

        for gt_person in gt_persons:

            for field in PERSON_FIELDS:

                gt_value = gt_person.get(
                    field
                )

                if not is_empty(
                    gt_value
                ):

                    field_name = (
                        f"person.{field}"
                    )

                    field_end_to_end[
                        field_name
                    ]["missing"] += 1


        # Record unmatched event itself

        error_rows.append({

            "issue_id":
                issue_id,

            "event_type":
                event_type,

            "field":
                "__event__",

            "ground_truth":
                json.dumps(
                    gt_record,
                    ensure_ascii=False
                ),

            "prediction":
                "",

            "error_type":
                "missing_event",

            "role":
                "",

            "event_match_score":
                ""
        })



    # UNMATCHED PREDICTED EVENTS

    for pred_i in unmatched_pred:

        pred_event = pred_events[
            pred_i
        ]

        error_rows.append({

            "issue_id":
                issue_id,

            "event_type":
                pred_event["event_type"],

            "field":
                "__event__",

            "ground_truth":
                "",

            "prediction":
                json.dumps(
                    pred_event["record"],
                    ensure_ascii=False
                ),

            "error_type":
                "spurious_event",

            "role":
                "",

            "event_match_score":
                ""
        })


    return {

        "issue_id":
            issue_id,

        "ground_truth_events":
            len(gt_events),

        "predicted_events":
            len(pred_events),

        "tp":
            tp,

        "fp":
            fp,

        "fn":
            fn,

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "event_metrics":
            event_metrics,

        "field_stats":
            dict(field_stats),

        "field_end_to_end":
            dict(field_end_to_end),

        "event_matches":
            event_matches,

        "unmatched_gt":
            unmatched_gt,

        "unmatched_pred":
            unmatched_pred,

        "error_rows":
            error_rows
    }



# FIND FILES

vlm_files = find_vlm_files()

hand_files = find_hand_files()


print("=" * 70)
print("REGDIA GROUND-TRUTH EVALUATION")
print("=" * 70)

print(
    f"\nVLM files found:          {len(vlm_files)}"
)

print(
    f"Hand-extracted files:     {len(hand_files)}"
)


issues_to_evaluate = sorted(
    set(hand_files.keys())
    &
    set(vlm_files.keys())
)


missing_vlm = sorted(
    set(hand_files.keys())
    -
    set(vlm_files.keys())
)


print(
    f"Issues to evaluate:       "
    f"{len(issues_to_evaluate)}"
)


if missing_vlm:

    print(
        "\nWARNING: hand-extracted files "
        "without matching VLM JSON:"
    )

    for issue_id in missing_vlm:

        print(
            f"  - {issue_id}"
        )



# EVALUATE ALL ISSUES

results = {}


for issue_id in issues_to_evaluate:

    print(
        f"\nEvaluating {issue_id}..."
    )


    gt_data = load_json(
        hand_files[issue_id]
    )

    pred_data = load_json(
        vlm_files[issue_id]
    )


    results[
        issue_id
    ] = evaluate_issue(
        issue_id,
        gt_data,
        pred_data
    )


    result = results[
        issue_id
    ]


    print(
        f"  GT events: "
        f"{result['ground_truth_events']}"
    )

    print(
        f"  Predicted: "
        f"{result['predicted_events']}"
    )

    print(
        f"  Matched: "
        f"{result['tp']}"
    )

    print(
        f"  FP: "
        f"{result['fp']}"
    )

    print(
        f"  FN: "
        f"{result['fn']}"
    )

    print(
        f"  P={result['precision']:.3f} "
        f"R={result['recall']:.3f} "
        f"F1={result['f1']:.3f}"
    )



# OVERALL EVENT METRICS

total_tp = sum(
    result["tp"]
    for result in results.values()
)

total_fp = sum(
    result["fp"]
    for result in results.values()
)

total_fn = sum(
    result["fn"]
    for result in results.values()
)


micro_precision = (
    total_tp
    /
    (total_tp + total_fp)
    if total_tp + total_fp
    else 0
)

micro_recall = (
    total_tp
    /
    (total_tp + total_fn)
    if total_tp + total_fn
    else 0
)

micro_f1 = (
    2
    * micro_precision
    * micro_recall
    /
    (
        micro_precision
        + micro_recall
    )
    if (
        micro_precision
        + micro_recall
    )
    else 0
)



# MACRO METRICS

if results:

    macro_precision = sum(
        result["precision"]
        for result in results.values()
    ) / len(results)

    macro_recall = sum(
        result["recall"]
        for result in results.values()
    ) / len(results)

    macro_f1 = sum(
        result["f1"]
        for result in results.values()
    ) / len(results)

else:

    macro_precision = 0
    macro_recall = 0
    macro_f1 = 0



# OVERALL RESULT

print("\n")
print("=" * 70)
print("OVERALL RESULT")
print("=" * 70)

print(
    f"""
Issues evaluated:       {len(results)}

Ground-truth events:    {total_tp + total_fn}
Predicted events:       {total_tp + total_fp}

Matched events:         {total_tp}
False positives:        {total_fp}
False negatives:        {total_fn}

MICRO
Precision:              {micro_precision:.4f}
Recall:                 {micro_recall:.4f}
F1:                     {micro_f1:.4f}

MACRO
Precision:              {macro_precision:.4f}
Recall:                 {macro_recall:.4f}
F1:                     {macro_f1:.4f}
"""
)



# PER-ISSUE TABLE

print("\n")
print("=" * 70)
print("PER-ISSUE RESULTS")
print("=" * 70)


header = (
    f"{'Issue':<30}"
    f"{'GT':>7}"
    f"{'Pred':>7}"
    f"{'TP':>7}"
    f"{'FP':>7}"
    f"{'FN':>7}"
    f"{'Prec.':>9}"
    f"{'Recall':>9}"
    f"{'F1':>9}"
)

print(header)
print("-" * len(header))


for issue_id, result in results.items():

    print(
        f"{issue_id:<30}"
        f"{result['ground_truth_events']:>7}"
        f"{result['predicted_events']:>7}"
        f"{result['tp']:>7}"
        f"{result['fp']:>7}"
        f"{result['fn']:>7}"
        f"{result['precision']:>9.3f}"
        f"{result['recall']:>9.3f}"
        f"{result['f1']:>9.3f}"
    )


# EVENT TYPE SUMMARY

overall_event_stats = defaultdict(
    lambda: {
        "ground_truth": 0,
        "predicted": 0,
        "tp": 0
    }
)


for result in results.values():

    for (
        event_type,
        metrics
    ) in result[
        "event_metrics"
    ].items():

        overall_event_stats[
            event_type
        ]["ground_truth"] += (
            metrics["ground_truth"]
        )

        overall_event_stats[
            event_type
        ]["predicted"] += (
            metrics["predicted"]
        )

        overall_event_stats[
            event_type
        ]["tp"] += (
            metrics["tp"]
        )


print("\n")
print("=" * 70)
print("EVENT-TYPE RESULTS")
print("=" * 70)


header = (
    f"{'Event type':<25}"
    f"{'GT':>7}"
    f"{'Pred':>7}"
    f"{'TP':>7}"
    f"{'FP':>7}"
    f"{'FN':>7}"
    f"{'Prec.':>9}"
    f"{'Recall':>9}"
    f"{'F1':>9}"
)

print(header)
print("-" * len(header))


for event_type, stats in sorted(
    overall_event_stats.items()
):

    gt = stats[
        "ground_truth"
    ]

    pred = stats[
        "predicted"
    ]

    tp = stats[
        "tp"
    ]

    fp = (
        pred - tp
    )

    fn = (
        gt - tp
    )

    p = (
        tp
        /
        (tp + fp)
        if tp + fp
        else 0
    )

    r = (
        tp
        /
        (tp + fn)
        if tp + fn
        else 0
    )

    f = (
        2 * p * r
        /
        (p + r)
        if p + r
        else 0
    )


    print(
        f"{event_type:<25}"
        f"{gt:>7}"
        f"{pred:>7}"
        f"{tp:>7}"
        f"{fp:>7}"
        f"{fn:>7}"
        f"{p:>9.3f}"
        f"{r:>9.3f}"
        f"{f:>9.3f}"
    )



# COMBINE FIELD STATISTICS

overall_fields = defaultdict(
    new_field_stats
)

overall_end_to_end = defaultdict(
    new_field_stats
)


for result in results.values():

    for field, stats in result[
        "field_stats"
    ].items():

        for key, value in stats.items():

            overall_fields[
                field
            ][key] += value


    for field, stats in result[
        "field_end_to_end"
    ].items():

        for key, value in stats.items():

            overall_end_to_end[
                field
            ][key] += value



# CONDITIONAL FIELD SUMMARY

print("\n")
print("=" * 70)
print("FIELD-LEVEL EXACT MATCH")
print(
    "(aligned events only)"
)
print("=" * 70)


header = (
    f"{'Field':<35}"
    f"{'Correct':>9}"
    f"{'Incorrect':>11}"
    f"{'Missing':>9}"
    f"{'Spurious':>10}"
    f"{'Both null':>10}"
    f"{'Accuracy':>10}"
)

print(header)
print("-" * len(header))


for field, stats in sorted(
    overall_fields.items()
):

    correct = stats[
        "correct"
    ]

    incorrect = stats[
        "incorrect"
    ]

    missing = stats[
        "missing"
    ]

    comparable = (
        correct
        + incorrect
        + missing
    )

    accuracy = (
        correct
        /
        comparable
        if comparable
        else 0
    )


    print(
        f"{field:<35}"
        f"{correct:>9}"
        f"{incorrect:>11}"
        f"{missing:>9}"
        f"{stats['spurious']:>10}"
        f"{stats['both_null']:>10}"
        f"{accuracy:>10.3f}"
    )



# END-TO-END FIELD SUMMARY

print("\n")
print("=" * 70)
print("FIELD-LEVEL END-TO-END RECOVERY")
print(
    "(all ground-truth events)"
)
print("=" * 70)


header = (
    f"{'Field':<35}"
    f"{'Correct':>9}"
    f"{'Incorrect':>11}"
    f"{'Missing':>9}"
    f"{'Spurious':>10}"
    f"{'Both null':>10}"
    f"{'Recovery':>10}"
)

print(header)
print("-" * len(header))


for field, stats in sorted(
    overall_end_to_end.items()
):

    correct = stats[
        "correct"
    ]

    incorrect = stats[
        "incorrect"
    ]

    missing = stats[
        "missing"
    ]

    gt_present = (
        correct
        + incorrect
        + missing
    )

    recovery = (
        correct
        /
        gt_present
        if gt_present
        else 0
    )


    print(
        f"{field:<35}"
        f"{correct:>9}"
        f"{incorrect:>11}"
        f"{missing:>9}"
        f"{stats['spurious']:>10}"
        f"{stats['both_null']:>10}"
        f"{recovery:>10.3f}"
    )



# SAVE ISSUE SUMMARY CSV

summary_csv = os.path.join(
    REPORT_BASE,
    "ground_truth_issue_summary.csv"
)


with open(
    summary_csv,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow([
        "issue_id",
        "ground_truth_events",
        "predicted_events",
        "tp",
        "fp",
        "fn",
        "precision",
        "recall",
        "f1"
    ])


    for issue_id, result in (
        results.items()
    ):

        writer.writerow([

            issue_id,

            result[
                "ground_truth_events"
            ],

            result[
                "predicted_events"
            ],

            result[
                "tp"
            ],

            result[
                "fp"
            ],

            result[
                "fn"
            ],

            result[
                "precision"
            ],

            result[
                "recall"
            ],

            result[
                "f1"
            ]
        ])



# SAVE EVENT TYPE CSV

event_csv = os.path.join(
    REPORT_BASE,
    "ground_truth_event_type_summary.csv"
)


with open(
    event_csv,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow([
        "event_type",
        "ground_truth",
        "predicted",
        "tp",
        "fp",
        "fn",
        "precision",
        "recall",
        "f1"
    ])


    for (
        event_type,
        stats
    ) in sorted(
        overall_event_stats.items()
    ):

        gt = stats[
            "ground_truth"
        ]

        pred = stats[
            "predicted"
        ]

        tp = stats[
            "tp"
        ]

        fp = pred - tp
        fn = gt - tp


        p = (
            tp
            /
            (tp + fp)
            if tp + fp
            else 0
        )

        r = (
            tp
            /
            (tp + fn)
            if tp + fn
            else 0
        )

        f = (
            2 * p * r
            /
            (p + r)
            if p + r
            else 0
        )


        writer.writerow([
            event_type,
            gt,
            pred,
            tp,
            fp,
            fn,
            p,
            r,
            f
        ])



# SAVE CONDITIONAL FIELD CSV

field_csv = os.path.join(
    REPORT_BASE,
    "ground_truth_field_summary.csv"
)


with open(
    field_csv,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow([
        "field",
        "correct",
        "incorrect",
        "missing",
        "spurious",
        "both_null",
        "accuracy"
    ])


    for field, stats in sorted(
        overall_fields.items()
    ):

        correct = stats[
            "correct"
        ]

        incorrect = stats[
            "incorrect"
        ]

        missing = stats[
            "missing"
        ]

        comparable = (
            correct
            + incorrect
            + missing
        )

        accuracy = (
            correct
            /
            comparable
            if comparable
            else 0
        )


        writer.writerow([
            field,
            correct,
            incorrect,
            missing,
            stats[
                "spurious"
            ],
            stats[
                "both_null"
            ],
            accuracy
        ])



# SAVE END-TO-END FIELD CSV

end_to_end_csv = os.path.join(
    REPORT_BASE,
    "ground_truth_field_end_to_end.csv"
)


with open(
    end_to_end_csv,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow([
        "field",
        "correct",
        "incorrect",
        "missing",
        "spurious",
        "both_null",
        "recovery"
    ])


    for field, stats in sorted(
        overall_end_to_end.items()
    ):

        correct = stats[
            "correct"
        ]

        incorrect = stats[
            "incorrect"
        ]

        missing = stats[
            "missing"
        ]

        gt_present = (
            correct
            + incorrect
            + missing
        )

        recovery = (
            correct
            /
            gt_present
            if gt_present
            else 0
        )


        writer.writerow([
            field,
            correct,
            incorrect,
            missing,
            stats[
                "spurious"
            ],
            stats[
                "both_null"
            ],
            recovery
        ])



# SAVE EVENT MATCHING DIAGNOSTICS

matches_csv = os.path.join(
    REPORT_BASE,
    "ground_truth_event_matches.csv"
)


with open(
    matches_csv,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.writer(
        f
    )

    writer.writerow([
        "issue_id",
        "event_type",
        "gt_index",
        "pred_index",
        "match_score",
        "gt_date",
        "pred_date",
        "gt_person",
        "pred_person",
        "gt_direction",
        "pred_direction",
        "gt_parish",
        "pred_parish",
        "date_evidence",
        "person_similarity",
        "direction_evidence"
    ])


    for issue_id, result in (
        results.items()
    ):

        # Need the source events again to make the
        # diagnostics readable.

        gt_data = load_json(
            hand_files[
                issue_id
            ]
        )

        pred_data = load_json(
            vlm_files[
                issue_id
            ]
        )

        gt_events = extract_events(
            gt_data
        )

        pred_events = extract_events(
            pred_data
        )


        for match in result[
            "event_matches"
        ]:

            gt_event = gt_events[
                match["gt_index"]
            ]

            pred_event = pred_events[
                match["pred_index"]
            ]


            gt_record = gt_event[
                "record"
            ]

            pred_record = pred_event[
                "record"
            ]


            gt_person = ""

            if gt_event[
                "persons"
            ]:

                gt_person = person_full_name(
                    gt_event[
                        "persons"
                    ][0]
                )


            pred_person = ""

            if pred_event[
                "persons"
            ]:

                pred_person = person_full_name(
                    pred_event[
                        "persons"
                    ][0]
                )


            evidence = match[
                "evidence"
            ]


            writer.writerow([

                issue_id,

                gt_event[
                    "event_type"
                ],

                match[
                    "gt_index"
                ],

                match[
                    "pred_index"
                ],

                match[
                    "match_score"
                ],

                get_nested_value(
                    gt_record,
                    "date.verbatim"
                ),

                get_nested_value(
                    pred_record,
                    "date.verbatim"
                ),

                gt_person,

                pred_person,

                gt_record.get(
                    "direction",
                    ""
                ),

                pred_record.get(
                    "direction",
                    ""
                ),

                gt_event.get(
                    "parish",
                    ""
                ),

                pred_event.get(
                    "parish",
                    ""
                ),

                evidence.get(
                    "date",
                    ""
                ),

                evidence.get(
                    "person_similarity",
                    ""
                ),

                evidence.get(
                    "direction",
                    ""
                )
            ])



# SAVE DETAILED ERROR CSV

errors_csv = os.path.join(
    REPORT_BASE,
    "ground_truth_errors.csv"
)


error_fields = [
    "issue_id",
    "event_type",
    "field",
    "ground_truth",
    "prediction",
    "error_type",
    "role",
    "event_match_score"
]


with open(
    errors_csv,
    "w",
    encoding="utf-8",
    newline=""
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=error_fields,
        extrasaction="ignore"
    )

    writer.writeheader()


    for result in results.values():

        for row in result[
            "error_rows"
        ]:

            writer.writerow(
                row
            )



# SAVE COMPLETE JSON

full_result_path = os.path.join(
    REPORT_BASE,
    "ground_truth_evaluation.json"
)


with open(
    full_result_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )



# FINISHED

print("\n")
print("=" * 70)
print("REPORTS SAVED")
print("=" * 70)


print(
    f"\nIssue summary:\n"
    f"{summary_csv}"
)

print(
    f"\nEvent-type summary:\n"
    f"{event_csv}"
)

print(
    f"\nConditional field summary:\n"
    f"{field_csv}"
)

print(
    f"\nEnd-to-end field summary:\n"
    f"{end_to_end_csv}"
)

print(
    f"\nEvent matching diagnostics:\n"
    f"{matches_csv}"
)

print(
    f"\nDetailed errors:\n"
    f"{errors_csv}"
)

print(
    f"\nComplete evaluation:\n"
    f"{full_result_path}"
)

print(
    "\nNo source JSON files were modified."
)

REGDIA GROUND-TRUTH EVALUATION

VLM files found:          52
Hand-extracted files:     3
Issues to evaluate:       3

Evaluating Num_2_09_01_1770...
  GT events: 93
  Predicted: 58
  TP: 9 FP: 49 FN: 84
  P=0.155 R=0.097 F1=0.119

Evaluating Num_31_31_07_1770...
  GT events: 119
  Predicted: 52
  TP: 5 FP: 47 FN: 114
  P=0.096 R=0.042 F1=0.058

Evaluating Num_3_16_01_1770...
  GT events: 101
  Predicted: 49
  TP: 21 FP: 28 FN: 80
  P=0.429 R=0.208 F1=0.280


OVERALL RESULT

Issues evaluated:       3

Ground-truth events:    313
Predicted events:       159

True positives:         35
False positives:        124
False negatives:        278

MICRO
Precision:              0.2201
Recall:                 0.1118
F1:                     0.1483

MACRO
Precision:              0.2266
Recall:                 0.1156
F1:                     0.1526



PER-ISSUE RESULTS
Issue                              GT   Pred     TP     FP     FN    Prec.   Recall       F1
----------------------------------------